# 🎨 Notebook 1: What is a Backend-for-Frontend (BFF)?

A **BFF** is a small API gateway dedicated to **one type of client** (mobile, web, smart TV, ...).

Instead of one giant 'universal' API that tries to serve every client, you build **several slim gateways**, each shaped for its audience.

### Analogy
Think of a restaurant. A *single buffet* tries to feed everyone — toddlers, vegans, bodybuilders — with the same food.
A *BFF* is a dedicated waiter per table who fetches **only what that table ordered**, plated the way they like it.


## 🛠️ Setup

```bash
cd 05-microservices/bff
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## The problem: one API for everyone

Mobile apps want **small** payloads (bandwidth matters).
Web apps can handle **rich** payloads.
Smart-TVs need **pre-rendered** text strings.

If one API serves all, every client either over-fetches or under-fetches.

In [ ]:
# Simulated 'universal' backend service
def get_user(user_id):
    return {'id': user_id, 'name': 'Ada', 'email': 'ada@example.com',
            'avatar_url': 'https://.../big.png', 'preferences': {'theme': 'dark'}}

def get_orders(user_id):
    return [{'id': 1, 'total': 42.0, 'items': [{'sku': 'A', 'qty': 2}]}]

def get_recommendations(user_id):
    return [{'sku': 'B', 'score': 0.9}, {'sku': 'C', 'score': 0.8}]

# A mobile client only wants: name + order count. But with a single 'universal' API it
# would have to call all three and ship a lot of bytes just to throw most of it away.
import json
universal_response = {'user': get_user(1), 'orders': get_orders(1), 'recs': get_recommendations(1)}
print('bytes over the wire:', len(json.dumps(universal_response)))


## The BFF solution

Build a gateway **per client type**. Each BFF calls the same downstream services but returns exactly what *that* client needs.

In [ ]:
def mobile_bff(user_id):
    """Tiny payload for mobile — just the headline numbers."""
    u = get_user(user_id)
    o = get_orders(user_id)
    return {'name': u['name'], 'order_count': len(o)}

def web_bff(user_id):
    """Rich payload for web dashboard — includes recommendations."""
    return {'user': get_user(user_id),
            'orders': get_orders(user_id),
            'recommendations': get_recommendations(user_id)}

import json
print('mobile bytes:', len(json.dumps(mobile_bff(1))))
print('web bytes:   ', len(json.dumps(web_bff(1))))


### Why this is better
- Each BFF is **owned by the team that owns the client** — no 'platform team bottleneck'.
- Mobile payloads shrink dramatically.
- Downstream services stay unchanged.

👉 Next notebook: a side-by-side latency/bandwidth comparison.